In [14]:
from rdflib import Graph,URIRef,Namespace,BNode,Literal
from rdflib.namespace import XSD,RDF
import pandas as pd
import argparse
from collections import defaultdict

In [4]:
graph = Graph()
print("Started loading graph...")
graph.parse('data/data.ttl', format="turtle")
print("Graph loaded successfully.")

Started loading graph...
Graph loaded successfully.


In [5]:
##EDITED

observation_type = URIRef("http://www.w3.org/ns/sosa/Observation")
rdf_type = URIRef("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")
observation_triples = Graph()
triples_with_no_observation = Graph()
subjects_has_observations = set()

In [6]:
##EDITED
#Find all triples with rdf:type sosa:Observation
for s,p,o in graph.triples((None, rdf_type, observation_type)):
    subjects_has_observations.add(s)


for triple in graph:
    s,p,o = triple
    if s in subjects_has_observations:
        observation_triples.add(triple)
    else:
        triples_with_no_observation.add(triple)
 

In [7]:
#for s, p, o in observation_triples:
    #print(f"Subject: {s}")
    #print(f"Predicate: {p}")
    #print(f"Object: {o}")
    #print("---")

In [8]:
length = len(subjects_has_observations)
fileno=0
count = 0
temp_graph = Graph()
for subject in subjects_has_observations:
    for pred, obj in observation_triples.predicate_objects(subject=subject):
        temp_graph.add((subject,pred,obj))
    
    count=count+1
    if count==5:
        count=0
        temp_graph.serialize(destination=f"timeseries_part{fileno}.ttl", format="turtle")
        fileno=fileno+1
        temp_graph = Graph()

In [42]:
#make new graph from existing dicitionaries
event_stream = "#EventStream"
#type_ns = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")
ldes = Namespace("https://w3id.org/ldes#")
base = Namespace("http://example.com/weights")
sosa = Namespace("http://www.w3.org/ns/sosa/")
tree = Namespace("https://w3id.org/tree#")
xsd = Namespace("http://www.w3.org/2001/XMLSchema#")
shapes = Namespace("http://example.com/shapes/")
ex = Namespace("http://example.org/")
dcat = Namespace("http://www.w3.org/ns/dcat#")
dc = Namespace("http://purl.org/dc/terms/")
prov = Namespace("http://purl.org/dc/terms/")
vsds = Namespace("https://implementatie.data.vlaanderen.be/ns/vsds-verkeersmetingen#")
shacl = Namespace("http://www.w3.org/ns/shacl#")

base = Namespace('https://example.com/')
obs = Namespace("https://example.com/observations/")
pageURL = Namespace('ldes_page')

In [43]:
#Linked Data Event Stream (LDES)
LDES_graph = Graph()

#LDES_graph.bind("a",type_ns)

LDES_graph.bind("ldes",ldes)
LDES_graph.bind("base",base)
LDES_graph.bind("sosa",sosa)
LDES_graph.bind("tree",tree)
LDES_graph.bind("xsd",xsd)
LDES_graph.bind("shapes",shapes)
LDES_graph.bind("ex",ex)
LDES_graph.bind("dcat",dcat)
LDES_graph.bind("dc",dc)
LDES_graph.bind("prov",prov)
LDES_graph.bind("vsds",vsds)
LDES_graph.bind("shacl",shacl)

LDES_graph.bind("base",base)
LDES_graph.bind("obs",obs)
LDES_graph.bind("pageURL",pageURL)


blank_node = BNode()

LDES_graph.add((base.observations, RDF.type,dcat.dataset))
LDES_graph.add((base.observations, dc.accessRights,dcat.public))
LDES_graph.add((base.observations, ldes.timestampPath,prov.generatedAtTime))
LDES_graph.add((base.observations, ldes.versionOfPath,dc.isVersionOf))
LDES_graph.add((base.observations, tree.shape,blank_node))
LDES_graph.add((blank_node, shacl.targetClass,vsds.Verkeerstelling))
LDES_graph.add((base.observations, tree.view,base.byPage))
LDES_graph.add((base.observations, dcat.endpointURL,URIRef(str(base))))

LDES_graph.add((obs.byPage, RDF.type, tree.node))
LDES_graph.add((obs.byPage, tree.viewDescription, obs.byPageDesc))
LDES_graph.add((obs.byPage, dcat.endpointURL, URIRef(str(pageURL))))




LDES_graph.add((obs.byPageDesc, RDF.type, tree.ViewDescription))
LDES_graph.add((obs.byPageDesc, tree.pageSize, Literal(5, datatype=XSD.integer)))



<Graph identifier=N4ea23a2b69e448d8a3033d2bee90fd2e (<class 'rdflib.graph.Graph'>)>

In [44]:
LDES_graph.serialize(destination="generated-data/ldes_graph.ttl", format="turtle")


<Graph identifier=N4ea23a2b69e448d8a3033d2bee90fd2e (<class 'rdflib.graph.Graph'>)>